In [1]:
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/merged_dataset
!rm -rf /content/test_dataset
!unzip -q "/content/drive/MyDrive/MRP/merged_dataset.zip" -d "/content/merged_dataset"
!unzip -q "/content/drive/MyDrive/MRP/test_dataset.zip" -d "/content/test_dataset"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/merged_dataset/merged_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=64,
    validation_split=0.2,
    subset="training",
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/merged_dataset/merged_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=64,
    validation_split=0.2,
    subset="validation",
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/test_dataset/test_dataset",
    labels="inferred",
    image_size=(256, 256),
    batch_size=64
)

Found 351500 files belonging to 2 classes.
Using 281200 files for training.
Found 351500 files belonging to 2 classes.
Using 70300 files for validation.
Found 18124 files belonging to 2 classes.


In [4]:
def dct_full_image(image):
    # Convert to float32
    image = tf.image.convert_image_dtype(image, tf.float32)

    # Apply 2D DCT per channel
    # DCT over width
    x = tf.signal.dct(image, type=2, norm='ortho')

    # DCT over height
    x = tf.signal.dct(tf.transpose(x, perm=[1, 0, 2]), type=2, norm='ortho')
    x = tf.transpose(x, perm=[1, 0, 2])

    # Magnitude
    x = tf.abs(x)

    # Normalize
    x_min = tf.reduce_min(x)
    x_max = tf.reduce_max(x)
    x = (x - x_min) / (x_max - x_min + 1e-8)

    return x


def dct_full_batch(batch):
    return tf.map_fn(
        dct_full_image,
        batch,
        fn_output_signature=tf.float32
    )

In [6]:
train_dct = train_ds.map(lambda x, y: (dct_full_batch(x), y), num_parallel_calls=tf.data.AUTOTUNE)
val_dct   = val_ds.map(lambda x, y: (dct_full_batch(x), y), num_parallel_calls=tf.data.AUTOTUNE)
test_dct  = test_ds.map(lambda x, y: (dct_full_batch(x), y), num_parallel_calls=tf.data.AUTOTUNE)

train_dct = train_dct.prefetch(tf.data.AUTOTUNE)
val_dct   = val_dct.prefetch(tf.data.AUTOTUNE)
test_dct  = test_dct.prefetch(tf.data.AUTOTUNE)

In [7]:
def build_dct_cnn(input_shape=(256, 256, 3)):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 2
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 3
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 4
        layers.Conv2D(256, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Classification head
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu', name="dct_feature_vector"),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

    return model

model = build_dct_cnn()
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32, 32, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dct_feature_vector (Dense)      │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 423,361 (1.61 MB)

 Trainable params: 422,401 (1.61 MB)

 Non-trainable params: 960 (3.75 KB)

In [8]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [9]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    train_dct,
    validation_data=val_dct,
    epochs=20,
    callbacks=[early_stop]
)

Epoch 1/20
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 743s 165ms/step - accuracy: 0.7305 - loss: 0.5395 - val_accuracy: 0.5764 - val_loss: 0.7929
Epoch 2/20
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 745s 169ms/step - accuracy: 0.8205 - loss: 0.4181 - val_accuracy: 0.6318 - val_loss: 1.1185
Epoch 3/20
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 729s 166ms/step - accuracy: 0.8699 - loss: 0.3207 - val_accuracy: 0.8562 - val_loss: 0.3358
Epoch 4/20
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 721s 164ms/step - accuracy: 0.8907 - loss: 0.2706 - val_accuracy: 0.6870 - val_loss: 1.7429
Epoch 5/20
4394/4394 ━━━━━━━━━━━━━━━━━━━━ 699s 159ms/step - accuracy: 0.9039 - loss: 0.2356 - val_accuracy: 0.7208 - val_loss: 1.9204


In [10]:
loss, acc = model.evaluate(val_dct)
print("Validation accuracy:", acc)

1099/1099 ━━━━━━━━━━━━━━━━━━━━ 122s 110ms/step - accuracy: 0.8562 - loss: 0.3358
Validation accuracy: 0.8562304377555847


In [11]:
loss, acc = model.evaluate(test_dct)
print("Validation accuracy:", acc)

284/284 ━━━━━━━━━━━━━━━━━━━━ 33s 115ms/step - accuracy: 0.6606 - loss: 0.7391
Validation accuracy: 0.6605606079101562


In [12]:
model.save("/content/drive/MyDrive/MRP/models/dct_cnn_baseline.h5")

In [13]:
# Note: After run:
# model = load_model("/content/drive/MyDrive/MRP/models/dct_cnn_baseline.h5")
# feature_extractor = tf.keras.Model(
#     inputs=model.input,
#     outputs=model.get_layer("dct_feature_vector").output
# )